# Spatiotemporal demand forecasting

This analysis forecasts hourly bookings by pickup location using a chronological holdout. It fills the location-date-hour calendar with zero-demand buckets before splitting, compares an XGBoost model with a same-location/same-hour previous-week baseline, and reports overall, high-demand-location, and busy-hour errors. Model predictors are known calendar and location attributes only; the baseline is a rolling one-week-ahead comparison and is scored only when its prior-week observation exists. High-demand locations and busy hours are identified from training-period demand only.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

# Locate the checked-in cleaned data whether the notebook is run from the repo root or this folder.
data_candidates = [
    Path("uber_ncr_analysis") / "ncr_rides_cleaned_for_dashboard.csv",
    Path("..") / "uber_ncr_analysis" / "ncr_rides_cleaned_for_dashboard.csv",
]
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    searched = ", ".join(str(path) for path in data_candidates)
    raise FileNotFoundError(f"Cleaned ride data was not found. Checked: {searched}")

rides = pd.read_csv(data_path, usecols=["Date", "hour", "Pickup Location", "Booking ID"])
rides["Date"] = pd.to_datetime(rides["Date"], errors="coerce").dt.normalize()
rides["hour"] = pd.to_numeric(rides["hour"], errors="coerce")
rides = rides.dropna(subset=["Date", "hour", "Pickup Location"])
rides = rides[rides["hour"].between(0, 23)].copy()
rides["hour"] = rides["hour"].astype("int8")
if rides.empty:
    raise ValueError("No rides with valid date, hour, and pickup location were found.")

# Count bookings once per location/date/hour. Calendar covariates are derived below,
# rather than included in this groupby, so they cannot fragment a demand bucket.
keys = ["Pickup Location", "Date", "hour"]
observed = rides.groupby(keys, observed=True).size().rename("demand").reset_index()
locations = sorted(rides["Pickup Location"].unique())
dates = pd.date_range(rides["Date"].min(), rides["Date"].max(), freq="D")
calendar = pd.MultiIndex.from_product(
    [locations, dates, range(24)], names=keys
).to_frame(index=False)
demand_df = calendar.merge(observed, on=keys, how="left", validate="one_to_one")
demand_df["demand"] = demand_df["demand"].fillna(0).astype("int32")

# Construct predictors entirely from attributes known at forecast time.
demand_df["weekday"] = demand_df["Date"].dt.dayofweek.astype("int8")
demand_df["is_weekend"] = demand_df["weekday"].isin([5, 6]).astype("int8")
demand_df["hour_sin"] = np.sin(2 * np.pi * demand_df["hour"] / 24)
demand_df["hour_cos"] = np.cos(2 * np.pi * demand_df["hour"] / 24)
year_length = np.where(demand_df["Date"].dt.is_leap_year, 366, 365)
annual_phase = 2 * np.pi * (demand_df["Date"].dt.dayofyear - 1) / year_length
demand_df["day_of_year_sin"] = np.sin(annual_phase)
demand_df["day_of_year_cos"] = np.cos(annual_phase)
demand_df["day_index"] = (demand_df["Date"] - dates[0]).dt.days.astype("int16")

# Reserve the final 20% of calendar dates; no date is split across train and test.
if len(dates) < 10:
    raise ValueError("At least 10 calendar dates are required for a chronological holdout.")
split_index = int(np.floor(0.8 * len(dates)))
train_dates, test_dates = dates[:split_index], dates[split_index:]
train_df = demand_df[demand_df["Date"].isin(train_dates)].copy()
test_df = demand_df[demand_df["Date"].isin(test_dates)].copy()

categorical_features = ["Pickup Location", "weekday"]
numeric_features = [
    "hour_sin", "hour_cos", "is_weekend",
    "day_of_year_sin", "day_of_year_cos", "day_index",
]
features = categorical_features + numeric_features
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            XGBRegressor(
                n_estimators=150,
                max_depth=7,
                learning_rate=0.1,
                random_state=42,
                n_jobs=-1,
                objective="reg:squarederror",
            ),
        ),
    ]
)
model.fit(train_df[features], train_df["demand"])
test_df["model_prediction"] = np.clip(model.predict(test_df[features]), 0, None)

# Match each row to the actual demand at the same location/hour exactly seven days earlier.
# The previous-week comparator is a rolling one-week-ahead baseline, not a fixed-origin
# multi-week forecast. Missing matches remain missing and are excluded from both models'
# paired comparison rather than being silently replaced with zero.
previous_week = demand_df[keys + ["demand"]].copy()
previous_week["Date"] += pd.Timedelta(days=7)
previous_week = previous_week.rename(columns={"demand": "previous_week_demand"})
test_df = test_df.merge(previous_week, on=keys, how="left", validate="one_to_one")
paired_rows = test_df["previous_week_demand"].notna()

# Define busy cohorts using training-period demand only, then freeze them for evaluation.
busy_location_count = max(1, int(np.ceil(0.25 * len(locations))))
busy_locations = set(
    train_df.groupby("Pickup Location")["demand"]
    .mean()
    .nlargest(busy_location_count)
    .index
)
busy_hour_count = max(1, int(np.ceil(0.25 * 24)))
busy_hours = set(
    train_df.groupby("hour")["demand"]
    .mean()
    .nlargest(busy_hour_count)
    .index
)
cohorts = {
    "Overall": pd.Series(True, index=test_df.index),
    "High-demand locations (top 25%)": test_df["Pickup Location"].isin(busy_locations),
    "Busy hours (top 25%)": test_df["hour"].isin(busy_hours),
}

results = []
for cohort_name, cohort_mask in cohorts.items():
    cohort_test = test_df.loc[cohort_mask]
    paired = cohort_test.loc[cohort_test["previous_week_demand"].notna()]
    results.append(
        {
            "Cohort": cohort_name,
            "Test rows": len(cohort_test),
            "Paired rows": len(paired),
            "Baseline coverage (%)": 100 * len(paired) / len(cohort_test),
            "Model MAE": mean_absolute_error(paired["demand"], paired["model_prediction"]),
            "Model RMSE": np.sqrt(mean_squared_error(paired["demand"], paired["model_prediction"])),
            "Previous-week MAE": mean_absolute_error(paired["demand"], paired["previous_week_demand"]),
            "Previous-week RMSE": np.sqrt(
                mean_squared_error(paired["demand"], paired["previous_week_demand"])
            ),
        }
    )

metrics = pd.DataFrame(results).set_index("Cohort")
print(f"Data: {data_path} ({len(rides):,} rides; {len(locations)} locations)")
print(f"Complete hourly grid: {len(demand_df):,} rows across {len(dates)} dates")
print(f"Train: {train_dates[0].date()} through {train_dates[-1].date()} ({len(train_dates)} dates)")
print(f"Test:  {test_dates[0].date()} through {test_dates[-1].date()} ({len(test_dates)} dates)")
print(f"High-demand locations: {len(busy_locations)}; busy hours: {sorted(busy_hours)}")
print("\nPaired holdout metrics (lower is better; errors are rides per location-hour):")
display(metrics.round(3))

overall = metrics.loc["Overall"]
winner = "XGBoost" if overall["Model MAE"] < overall["Previous-week MAE"] else "Previous-week baseline"
print(
    f"Overall MAE is lower for the {winner}. The previous-week baseline covers "
    f"{overall['Baseline coverage (%)']:.1f}% of test rows; model and baseline errors "
    "above use the same matched rows."
)

## Reading the results

The final 20% of unique dates form one contiguous holdout. The completed location-date-hour grid makes zero bookings explicit, so the target is not restricted to hours in which a ride happened to occur. MAE is the average absolute error and RMSE penalizes larger misses more heavily; both are in rides per location-hour. The previous-week baseline requires an exact match seven calendar days earlier at the same pickup location and hour. Baseline coverage is shown for every cohort, and both methods' reported errors use the same matched rows. The model uses only location and calendar features that are known for the forecast date; it does not use held-out demand as a predictor. "High-demand locations" are the top quarter of locations by mean hourly demand in training, while "busy hours" are the six hour-of-day values with highest training-period mean demand. These cohorts are fixed before inspecting holdout outcomes.